# Lesson 8: South African Market Model

In [ ]:
lesson_number = 8
print(f'lesson{lesson_number}')

In [ ]:
#@title Connect to Google Drive {display-mode:"form"}
CONNECT_TO_DRIVE = False #@param {type:"boolean"}

import os

if CONNECT_TO_DRIVE:
    from google.colab import drive
    # Mount Google Drive
    drive.mount('/content/drive')

    # Define the desired working directory path
    working_dir = f'/content/drive/MyDrive/ich-modelling-2026'
    lesson_folder = f'lesson-{lesson_number}'
    lesson_dir = os.path.join(working_dir, lesson_folder)

    # Create the working directory if it doesn't exist
    if not os.path.exists(working_dir):
        os.makedirs(working_dir)
        print(f"Directory '{working_dir}' created.")
    else:
        print(f"Directory '{working_dir}' already exists.")

    # Create the lesson directory if it doesn't exist
    if not os.path.exists(lesson_dir):
        os.makedirs(lesson_dir)
        print(f"Directory '{lesson_dir}' created.")
    else:
        print(f"Directory '{lesson_dir}' already exists.")

    # Change the current working directory to the lesson directory
    os.chdir(lesson_dir)

    print(f"Current working directory: {os.getcwd()}")
else:
    print("Not connecting to Google Drive.")

In [ ]:
#@title Install Packages {display-mode:"form"}
INSTALL_PACKAGES = False #@param {type:"boolean"}

import os

# Check if packages have already been installed in this session to prevent re-installation
if INSTALL_PACKAGES and not os.environ.get('PYPSA_PACKAGES_INSTALLED'):
  !pip install git+https://github.com/pypsa/pypsa
  !pip install pypsa[excel] 
  !pip install folium mapclassify cartopy gurobipy
  os.environ['PYPSA_PACKAGES_INSTALLED'] = 'true'
elif not INSTALL_PACKAGES:
  print("Skipping package installation.")
else:
  print("PyPSA packages are already installed for this session.")

In [ ]:
#@title Download the file for this notebook {display-mode:"form"}
DOWNLOAD_FILE = False #@param {type:"boolean"}

if DOWNLOAD_FILE:
    !gdown "https://drive.google.com/uc?id=1My8j2qRcjjhVhC5bL657oYTk7-OKDxkE"

else:
    print("Skipping file download.")

In [ ]:
import gurobipy as gp
import pypsa
import pandas as pd
import linopy
import plotly.graph_objects as go

import linopy.solvers
linopy.solvers.gurobipy = gp
pypsa.options.api.new_components_api = True

pd.set_option('plotting.backend', 'plotly')

start = pd.Timestamp('2026-04-01 00:00:00')
end = start + pd.Timedelta(days=7) - pd.Timedelta(hours=1)

unit_commitment = True # True or False for unit commitment of dispatchable generators (coal and gas in this case)

solver_name = 'gurobi'  # 'gurobi' or 'highs'

solver_options={
        'TimeLimit': 600,        # 10 minutes
        'MIPGap': 0.02,          # Or accept 2% gap (0.02)
        'LogToConsole': 1 
        }

n = pypsa.Network('n-02-za-network-v3.xlsx')

n.set_snapshots(pd.date_range(start, end, freq='h'))

dispatchable_carriers = ['coal','ocgt_diesel','ocgt_avf','sasol_gas'] 
n.generators.static.loc[n.generators.static.carrier.isin(dispatchable_carriers), 'committable'] = unit_commitment

# Cluster the network spatially by bus location, and add load shedding to the clustered network
busmap = n.buses.static['location'].to_dict()  
n_clustered = n.cluster.spatial.cluster_by_busmap(busmap=busmap)

# n_clustered.optimize.add_load_shedding(marginal_cost=15e5)
# n_clustered.sanitize()


n_clustered.optimize(solver_name=solver_name, include_objective_constant = False)



In [ ]:
n_clustered.generators.dynamic.p.plot()